# 04. ESMFold로 접고 pLDDT 해석하기 🟡 (Colab GPU 권장)

**등급**: 🟡 Colab(T4 이상 GPU) · **다운로드**: `facebook/esmfold_v1` 가중치 약 8.4 GB (Colab 런타임에 저장됨) · **실측 시간**: Colab T4에서 코드 셀 전체 약 1.5분(가중치가 이미 내려받아진 재실행; 모델 로드 약 40초), 내려받기까지 포함한 첫 실행은 약 3분(로드 순서를 바꾸기 전 코드로 잰 값). 단일 측정이다.

> ✅ **검증 범위**: Colab T4 런타임(GPU 15.6 GB, RAM 약 12 GB, Python 3.13, torch 2.14.1, transformers 5.22.1, accelerate 1.15.0, biopython 1.88)에서 이 노트북의 코드 셀 7개를 처음부터 끝까지 오류 없이 실행했다(2026-11-06). 같은 코드를 다시 실행해도 출력한 수치가 모두 같았다. 단일 GPU·단일 환경이므로 다른 버전·장치에서는 달라질 수 있다.
>
> ⚠️ **메모리**: 로드 직후 fp32 전체가 GPU에 올라가 순간 최대 약 14.1 GB를 쓴다(T4 15.6 GB에서 여유가 약 1.5 GB). 가중치를 CPU로 먼저 불러오는 방식은 RAM 12 GB 런타임에서 불안정했다(3회 중 1회만 성공, 2회는 OOM으로 커널이 죽었다). 그래서 `device_map={"": 0}`으로 GPU에 바로 올린다. GPU 메모리가 더 작은 장치(예: 8 GB)에서는 이 노트북이 그대로 돌지 않는다.

**목표**: (1) 서열에서 구조를 예측하고 (2) 잔기별 **pLDDT**를 읽고 (3) 실험 구조와 비교(RMSD)하고 (4) 대조 서열(섞은 서열, 저복잡도 서열)에서 신뢰도가 어떻게 달라지는지 본다.

참고: 2026년에 ESM 팀(Biohub)이 ESMFold2를 공개했다 (모델 카드·프리프린트는 모듈 README 참고). 가중치가 훨씬 커서 이 노트북은 ESMFold v1을 쓴다.

In [ ]:
# Colab에서만 필요 (로컬에서는 SETUP.md의 환경을 사용):
# !pip -q install transformers accelerate biopython

import io
import time
import urllib.request
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch
from Bio.PDB import PDBParser, Superimposer
from transformers import EsmForProteinFolding

warnings.filterwarnings("ignore")
assert torch.cuda.is_available(), "GPU 런타임이 필요합니다 (Colab: 런타임 > 런타임 유형 변경 > T4 GPU)"
DEVICE = torch.device("cuda")
torch.backends.cuda.matmul.allow_tf32 = True

## 1. 모델 로드
- `device_map={"": 0}`로 가중치를 GPU에 바로 올린다. CPU로 먼저 불러오면 fp32 전체(약 14 GB)가 Colab 무료 런타임의 RAM(약 12 GB)을 넘겨 커널이 죽을 수 있다.
- ESM-2 backbone을 fp16으로 두어 메모리를 줄인다 (`model.esm = model.esm.half()`). 로드 직후에는 fp32 전체가 GPU에 있어 순간 최대 약 14 GB를 쓴다 (T4 15 GB에서 한계에 가깝다).
- 긴 서열의 메모리를 줄이도록 chunk 크기를 `model.trunk.set_chunk_size(64)`로 설정한다.
- 로드 보고서에 `esm.contact_head.regression.*`가 `MISSING`으로 나오지만 무시해도 된다. ESMFold의 접기 경로(`modeling_esmfold.py`)는 접촉 예측 헤드를 호출하지 않는다(transformers 5.22.1 소스에서 확인).

In [ ]:
from huggingface_hub import snapshot_download

# 리비전을 고정하고 로컬 경로에서 불러온다 (저장소 ID로 열면 .bin 전용 저장소에서 자동 변환 PR 파일이 섞인 적이 있다: 모듈 05).
# pytorch_model.bin은 pickle이다. 공식 facebook 저장소(MIT)의 고정 리비전만 쓰고, 로딩은 transformers의 weights_only 경로에 맡긴다.
ESMFOLD_REV = "75a3841ee059df2bf4d56688166c8fb459ddd97a"
ESMFOLD_DIR = snapshot_download("facebook/esmfold_v1", revision=ESMFOLD_REV)          # 약 8.4 GB
# 가중치를 CPU를 거치지 않고 GPU로 바로 올린다. CPU로 먼저 불러오면 fp32 전체(약 14 GB)가 RAM 12 GB를 넘겨 커널이 OOM으로 죽을 수 있다(T4 런타임에서 실제로 겪었다).
model = EsmForProteinFolding.from_pretrained(ESMFOLD_DIR, device_map={"": 0}).eval()
model.esm = model.esm.half()                                                          # 이후 ESM-2 backbone만 fp16으로 (GPU 위에서 매개변수별로 변환)
model.trunk.set_chunk_size(64)
print("params (M):", round(sum(p.numel() for p in model.parameters()) / 1e6))

## 2. 접기 함수와 pLDDT 추출
`model.infer(seq)`가 원시 출력 dict를 반환한다. `plddt`의 shape은 `(1, L, 37)`(atom37별)이고, 잔기 값은 **CA 원자(인덱스 1)**를 쓴다.
HF 구현의 pLDDT는 **0~1 범위**일 수 있으므로(범주형 혼합의 평균) 0~100 척도로 맞춰 표시한다.

In [ ]:
@torch.no_grad()
def fold(seq):
    out = model.infer(seq)
    plddt = out["plddt"][0, :, 1].float().cpu().numpy()          # CA 원자의 pLDDT
    assert plddt.max() <= 1.0 + 1e-3, "pLDDT 척도가 0~1이 아닙니다 — 라이브러리 버전이 바뀌었는지 확인하세요"
    plddt = plddt * 100                                           # 0~100 척도로 (HF 구현은 0~1을 반환한다: 소스 확인)
    pae = out["predicted_aligned_error"][0].float().cpu().numpy() # (L, L) 잔기쌍 정렬 오차 예측 (단위·척도는 모델 문서 확인†)
    pdb = model.output_to_pdb(out)[0]
    ptm = float(out["ptm"])
    torch.cuda.empty_cache()
    return pdb, plddt, ptm, pae


UBQ = "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"      # PDB 1UBQ, 76 aa
GB1 = "MTYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE"                            # PDB 1PGA, 56 aa
rng = np.random.RandomState(0)
SEQS = {
    "ubiquitin (1UBQ)": UBQ,
    "protein G B1 (1PGA)": GB1,
    "ubiquitin shuffled": "".join(rng.permutation(list(UBQ))),                               # 조성은 같고 순서만 섞음
    "low complexity (GS)n": "GS" * 38,
}
folded = {}
for name, s in SEQS.items():
    folded[name] = fold(s)
    pdb, pl, ptm, pae = folded[name]
    print(f"{name:24s} L={len(s):3d}  mean pLDDT={pl.mean():5.1f}  min={pl.min():5.1f}  pTM={ptm:.2f}  mean PAE={pae.mean():.2f}")

## 3. 잔기별 pLDDT

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
for name, (pdb, pl, ptm, pae) in folded.items():
    ax.plot(pl, label=name, lw=1.5)
ax.axhline(70, color="gray", ls="--", lw=0.8)
ax.axhline(50, color="gray", ls=":", lw=0.8)
ax.set_xlabel("residue")
ax.set_ylabel("pLDDT (CA)")
ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

### PAE (잔기쌍 정렬 오차 예측)
pLDDT는 잔기별 국소 신뢰도이고, PAE는 **두 잔기의 상대 배치**에 대한 신뢰도다. 도메인 사이 배치는 pLDDT만으로 알 수 없고 PAE로 본다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, name in zip(axes, ["ubiquitin (1UBQ)", "protein G B1 (1PGA)", "ubiquitin shuffled"]):
    im = ax.imshow(folded[name][3], cmap="viridis_r")
    ax.set_title(name, fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

## 4. 실험 구조와 비교 (CA RMSD)
RCSB에서 1UBQ, 1PGA 구조를 내려받아 예측 구조와 CA 원자로 겹쳐 RMSD를 잰다. 서열 길이가 같고 잔기 순서가 같다는 가정(두 구조 모두 해당)이다.

In [ ]:
def ca_atoms(structure):
    return [r["CA"] for r in structure[0]["A"] if r.id[0] == " " and "CA" in r]


def rmsd_to_experiment(pid, pred_pdb):
    path = f"{pid}.pdb"
    for attempt in range(3):                                      # RCSB 연결이 일시적으로 끊길 수 있다 (Colab 실행에서 한 번 겪었다)
        try:
            urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pid}.pdb", path)
            break
        except OSError:
            if attempt == 2:
                raise
            time.sleep(3)
    exp = PDBParser(QUIET=True).get_structure("exp", path)
    pred = PDBParser(QUIET=True).get_structure("pred", io.StringIO(pred_pdb))
    pred_ca = [r["CA"] for r in pred[0].get_residues() if "CA" in r]
    exp_ca = ca_atoms(exp)
    assert len(exp_ca) == len(pred_ca), (len(exp_ca), len(pred_ca))
    sup = Superimposer()
    sup.set_atoms(exp_ca, pred_ca)
    return sup.rms


for name, pid in [("ubiquitin (1UBQ)", "1UBQ"), ("protein G B1 (1PGA)", "1PGA")]:
    print(f"{name:22s} CA RMSD vs experiment: {rmsd_to_experiment(pid, folded[name][0]):.2f} Å")

## 5. 점 돌연변이의 영향 — 한 서열의 변이 몇 개
유비퀴틴의 무작위 점돌연변이 8개를 접고 평균 pLDDT 변화를 본다.

In [ ]:
AA = "ACDEFGHIKLMNPQRSTVWY"
wt_pl = folded["ubiquitin (1UBQ)"][1]
W = 5                                                                 # 변이 위치 ±5 잔기 창
rows = []
for _ in range(8):
    i = int(rng.randint(len(UBQ)))
    new = rng.choice([a for a in AA if a != UBQ[i]])
    mut = UBQ[:i] + new + UBQ[i + 1:]
    _, pl, _, _ = fold(mut)
    lo, hi = max(0, i - W), min(len(UBQ), i + W + 1)
    rows.append((f"{UBQ[i]}{i + 1}{new}", pl.mean() - wt_pl.mean(), pl[lo:hi].mean() - wt_pl[lo:hi].mean()))
for m, d_all, d_loc in rows:
    print(f"{m:6s} Δ 평균 pLDDT(전체) = {d_all:+.2f} | Δ 평균 pLDDT(변이 ±{W}) = {d_loc:+.2f}")

## 6. 해석 가이드
**관찰한 결과** (Colab T4, 단일 서열·단일 모델 가중치, 같은 코드 재실행에서 동일):
- 자연 단백질: 유비퀴틴 평균 pLDDT 90.5, GB1 89.0이고 실험 구조와의 CA RMSD는 0.84 Å, 0.42 Å였다. 두 단백질 모두 사전학습 데이터에 들어 있을 가능성이 있어(평가–사전학습 겹침, 모듈 01)† 이 수치만으로 처음 보는 단백질에서도 잘 맞는다고 말할 수 없다.
- 대조군: 섞은 서열은 평균 pLDDT 42.8(pTM 0.18, 평균 PAE 19.7), `(GS)n` 저복잡도 서열은 27.3(pTM 0.09, 평균 PAE 19.9)이었다. 자연 단백질의 pTM은 0.83·0.76, 평균 PAE는 3.0·2.8였다. 다만 **낮은 pLDDT = 무질서의 증명이 아니라 모델이 확신하지 못한다는 뜻**이다.
- 점돌연변이 8개: 전체 평균 pLDDT 변화는 −0.95~+0.29, 변이 ±5 창은 −1.98~+0.41로 대부분 ±1 안쪽이었다. L69T는 창 변화(−1.98)가 전체 변화(−0.94)의 두 배라 국소 효과가 전체 평균에 희석됨을 보여 준다. 표본이 8개뿐이고 pLDDT 변화는 변이의 기능적 영향을 직접 측정하지 않는다.

**읽는 법**
- pLDDT 구간별 의미는 모델 문서의 규약을 확인한다. 흔히 70 이상을 비교적 신뢰, 50 미만을 낮은 신뢰로 보지만 **척도(0~1 vs 0~100)를 먼저 확인**한다.
- pTM·PAE는 도메인 배치와 전체 폴드의 신뢰도를 다룬다(이 노트북은 pTM, 평균 PAE, PAE 행렬을 출력한다). pLDDT만으로 도메인 상대 배치는 알 수 없다.
- 단일 서열 모델(ESMFold)은 MSA 기반 모델과 다른 오류 패턴을 보인다. 사전학습 데이터에 포함된 단백질에서는 특히 잘 맞을 수 있다 (평가–사전학습 겹침, 모듈 01).

## 7. 한계와 과제
- 구조 하나를 정확히 예측한다는 것이 변이 효과, 동역학, 상호작용을 이해한다는 뜻은 아니다.
- 과제: [../exercises.md](../exercises.md) 과제 5 (ProteinGym 소형 assay에서 pLDDT 변화와 DMS 점수의 상관).